# Journey 3 — Configure SWAN declaratively with YAML

**Learning goals:** Understand how a serialised model run relates to Python configuration and why declarative files help with sharing and reproducibility.

**Prerequisites:** [Journey 2](journey_02_swan_procedural.ipynb).


!!! note
    This journey demonstrates configuration and workspace generation. The documentation build does not execute notebook cells or require a SWAN binary. Execution prerequisites are called out separately.


## 1. Represent configuration as data

A declarative file records the model run’s identity, period, output directory, and plugin configuration without requiring the reader to reconstruct the Python object graph first.


In [ ]:
from pathlib import Path
import yaml

from rompy.model import ModelRun
from rompy.core.time import TimeRange
from rompy_swan.config import SwanConfig
from rompy_swan.grid import SwanGrid
from rompy_swan.components.cgrid import REGULAR
from rompy_swan.subcomponents.readgrid import GRIDREGULAR
from rompy_swan.subcomponents.spectrum import SPECTRUM

# Build the same minimal model shape as Journey 2, then serialise it.
grid = SwanGrid(x0=115.0, y0=-32.0, rot=0.0, dx=0.25, dy=0.25, nx=9, ny=7)
cgrid = REGULAR(
    grid=GRIDREGULAR(
        xp=grid.x0, yp=grid.y0, alp=grid.rot,
        xlen=grid.xlen, ylen=grid.ylen,
        mx=grid.nx - 1, my=grid.ny - 1,
    ),
    spectrum=SPECTRUM(mdc=36, flow=0.04, fhigh=1.0),
)
config = SwanConfig(cgrid=cgrid)

document = {
    "run_id": "swan_first_run",
    "period": {
        "start": "2023-01-01T00:00:00",
        "end": "2023-01-01T06:00:00",
        "interval": "1h",
    },
    "output_dir": "swan_journey_workspace",
    "config": config.model_dump(exclude_none=True, exclude={"template", "checkout"}),
}
config_text = yaml.safe_dump(document, sort_keys=False)
print(config_text)

loaded_run = ModelRun(**yaml.safe_load(config_text))
print("loaded config:", loaded_run.config.model_type)


The compact document above is a teaching representation. A runnable SWAN YAML file must include the model-specific component tree and valid data interfaces. The repository’s [existing declarative example](example_declarative.ipynb) shows that complete form.


In [ ]:
# A real workflow loads the complete YAML document and constructs ModelRun.
from rompy.model import ModelRun

# run = ModelRun(**yaml.safe_load(Path("example_declarative.yml").read_text()))
# run.generate()


## 2. Compare procedural and declarative workflows

Both forms describe the same conceptual object:

| Concept | Procedural form | Declarative form |
| --- | --- | --- |
| Period | `TimeRange(...)` | `period:` mapping |
| SWAN setup | component objects | nested YAML mappings |
| Reuse | Python functions | copied/versioned YAML |
| Validation | Pydantic at construction | Pydantic while loading |
| Execution | `run(backend=...)` | the same `ModelRun` method |


## Checkpoint

YAML is not a different model engine. It is another representation of the same validated Rompy configuration. This makes configurations easier to review, share, and submit to a separate runtime.

**Next:** [Prepare SWAN grids and input data](journey_04_swan_data.ipynb).

**Further reading:** [Rompy configuration deep dive](https://rom-py.github.io/rompy/configuration_deep_dive/) and [schema advantages](https://rom-py.github.io/rompy/schema_advantages/).
